In [4]:
##### 250910 미션 해설!! #####

from pymongo import MongoClient

# 접속 혹은 접근 가능한 MongoDB 주소 (mongodb의 문법을 사용해야하기 때문에 앞에 mongodb 적음!)로 커넥트
client = MongoClient("mongodb://localhost:27017")
# sample_mflix로 접속 (대괄호표기법 & 온점표기법 모두 사용 가능)
# client.sample_mflix # 온점표기법 -> 파일명 안에 온점 포함 // 숫자로 시작하는 경우는 온점표기법 사용 불가! 
# (온점표기법과 파일명 중첩되는데 구분할 방법 없음)
db = client["sample_mflix"] # 대괄호 표기법
# db는 컴퓨터 메모리 공간 안에 있는 컬렉션 주소를 갖고있을 뿐. 실제로 그 안의 문서의 내용을 가지고 있지는 않음.
# 그래서 print(db) 입력하면 컬렉션에 대한 주소만 출력됨. 해당 컬렉션 값을 실제로 보고 싶다면 db.list_collection_names() 입력해야 함

users = db.users
comments = db.comments
movies = db.movies

print("ok", db.name, "collections : ", [c for c in db.list_collection_names() if c in {"users", "comments", "movies"}])
# [c for c in db.list_collection_names()] 
# for in 문은 실행문임. 이게 print구문 안에 들어가면 출력이 안 됨.
# -> 반복문에 대한 실행결과값을 출력하려면 for in 문 앞에 출력값의 이름을 입력해야 함. 
### print 내의 for in 문 실행/출력 구조!!
# 1) for c in db.list_collection_names() → db.list_collection_names()이 반환한 컬렉션 이름들의 리스트를 하나씩 꺼내면서, 그 값을 c라는 변수에 담음.
# 2) c for ... → 그 변수 c를 그대로 결과 리스트에 넣음.

# set() 중복값을 허용하지 않는 자료구조 형태({} 사용) ->{"users", "comments", "movies"}

ok sample_mflix collections :  ['movies', 'users', 'comments']


In [12]:
### 1) 충성 사용자 리포트
# • 이용자와 댓글 데이터를 결합해, 총 댓글 수가 10개 이상인 이용자만 추립니다.
# • 각 이용자에 대해 다음 지표를 계산해 주세요:
#     ◦ commentsCount(총 댓글 수)
#     ◦ avgTextLen(댓글 평균 길이, null/missing은 길이 0으로 간주)
#     ◦ lastCommentDate(가장 최근 댓글 시점)
# • 결과에는 **이름(name), 이메일(email)**만 식별 정보로 포함하고, 
# 위 활동 지표 **내림차순(많이·길게·최근 순)**으로 정렬해 상위 이용자가 누구인지 한눈에 보이도록 합니다.

# 유저 & 커멘트 연결 // 유저 -> 이메일 // 커멘트 -> text 길이 순으로 코드 작성할 순서 먼저 짜보기!

# users.aggregate([{},{},{}])의 형태로 작성되는 형태인데 안에 샤드값이 많아지면 가독성이 떨어지므로 pipeline으로 따로 빼서 씀!

loyal_users_pipeline = [
    {
        "$lookup": {
            "from": "comments",
            "localField": "email",
            "foreignField": "email",
            "as": "c",
        }
    },
    {
        "$addFields" : {
            "commentsCount": {"$size" : "$c"},  # $addFields => add할 필드명 작성 : {필드에 넣을 데이터를 추출하는 연산자 또는 계산식}
            "avgTextLen" : {
                "$avg" : {
                    "$map" : {                  # $map => 기존 배열을 변환해서 새로운 배열을 만들어주는 연산자
                        "input" : "$c",         # input : 변환할 원본 배열
                        "as" : "x",             # as ; 배열 요소를 가르킬 변수 이름
                        "in" : {"$strLenCP" : {"$ifNull" : ["$$x.text", ""]}}   # in : 각 요소마다 적용할 표현식(계산식)
                         # $map 안에서 as로 지정한 변수는 시스템 변수로 취급되기 때문에 접근할 때 반드시 $$를 붙여야 함!
                        # $ => 현재 문서(document)의 필드 접근 // $$ => 집계 파이프라인 내부에서 정의된 변수 접근
                    }                                                         
                }
            },
            
                "lastCommentDate" : {"$max" : "$c.date"}
            }
        
    },
    {"$match" : {"commentsCount" : {"$gte" : 10}}},
    {"$sort" : {"commentsCount" : -1, "avgTextLen" : -1, "lastCommentDate" : -1}},
    {
        "$project" : {
            "_id" : 0,
            "name" : 1,
            "email" : 1,
            "commentsCount" : 1,
            "avgTextLen" : 1,
            "lastCommentDate" : 1
        }
    }
]

# loyal_users = users.aggregate(loyal_users_pipeline)
# print(loyal_users)  => 얘도 주소값으로 출력되고 실제 값이 출력되진 않음. list() 넣기!
loyal_users = list(users.aggregate(loyal_users_pipeline))
len(loyal_users), loyal_users[0:5]  # -> 데이터 디지게 많아서 출력값 제한(슬라이스) 안 걸면 파이썬이  출력 거부함. (len() 걸어보니 183개 나옴)
# print 적은 것보다 안 적은게 가독성이 더 좋음! 싱기


### SQL & NoSQL -> 현업 실무 개발자들은 JOIN을 별로 좋아하지 않음. 
# 서로 다른 데이터 테이블 혹은 컬렉션을 하나로 연결해서 가져와 처리하려면 해당 데이터의 양이 방대해짐.
# 방대해진 데이터를 처리하기 위한 하드웨어에 많은 무리가 가고, 기계의 성능과 사양에 대한 제약이 발생됨.

(183,
 [{'name': 'Mace Tyrell',
   'email': 'roger_ashton-griffiths@gameofthron.es',
   'commentsCount': 331,
   'avgTextLen': 152.49244712990938,
   'lastCommentDate': datetime.datetime(2017, 8, 2, 1, 37, 26)},
  {'name': 'Missandei',
   'email': 'nathalie_emmanuel@gameofthron.es',
   'commentsCount': 327,
   'avgTextLen': 153.0948012232416,
   'lastCommentDate': datetime.datetime(2017, 9, 11, 16, 52, 51)},
  {'name': 'The High Sparrow',
   'email': 'jonathan_pryce@gameofthron.es',
   'commentsCount': 315,
   'avgTextLen': 152.68253968253967,
   'lastCommentDate': datetime.datetime(2017, 7, 10, 4, 58, 23)},
  {'name': 'Sansa Stark',
   'email': 'sophie_turner@gameofthron.es',
   'commentsCount': 308,
   'avgTextLen': 153.42532467532467,
   'lastCommentDate': datetime.datetime(2017, 5, 2, 16, 29, 50)},
  {'name': 'Rodrik Cassel',
   'email': 'ron_donachie@gameofthron.es',
   'commentsCount': 305,
   'avgTextLen': 151.75737704918032,
   'lastCommentDate': datetime.datetime(2017, 9, 2, 6

In [14]:
### 2) 영화 인사이트 리포트
# • 영화 컬렉션만으로 한 번의 파이프라인에서 다음을 동시에 산출해 주세요.
#     1. 최신 영화 5편: 제목(title)과 연도(year) 목록
#     2. 고평점 영화 개수: imdb.rating ≥ 8 작품의 총 개수(숫자 1개)
#     3. 장르별 상위 10: 장르별 작품 수를 집계해 상위 10개를 많이 나온 순으로 정렬
#     4. 연도별 평균 평점(최근 10년): 최신 연도를 기준으로 최근 10개 연도의 평균 평점을 산출(연도-오름차순 표기)출력 형식 가이드
# • 충성 사용자 리포트: 리스트(배열) 형태. 각 원소는 {name, email, commentsCount, avgTextLen, lastCommentDate}.
# • 영화 인사이트 리포트: 단일 문서(오브젝트) 형태.
#     ◦ 키는 latest5, highRatedCount, genresTop10, yearlyAvgRecent10로 구성.
#     ◦ highRatedCount는 숫자 하나로 제출


movies_report_Pipeline = [
    {
        "$facet" : {
            "latest5" : [                                           # $facet : 
                {"$limit" : 5},                                     # {결과문서 안에서 배열로 나오는 필드명 : [facet에 적용할 aggregation pipeline stage]}
                {"$sort" : {"year" : -1}},                          
                {"$project" : {"_id" : 0, "title" : 1, "year" : 1}}
            ],
            "highRatedCount" : [
                {"$match" : {"imdb.rating" : {"$gte" : 8}}},
                {"$count" : "count"}
            ],
            "genresTop10" : [
                {"$unwind" : "$genres"},
                {"$group" : {"_id" : "$genres", "count" : {"$sum" : 1}}},
                {"$sort" : {"count" : -1}},
                {"$limit" : 10}
            ],
            "yearlyAvgRecent10" : [
                {"$group" : {"_id" : "$year", "avgRating" : {"$avg" : "$imdb.rating"}}},  # 위에서는 단순히 값을 비교했지만 여기서는 사칙연산을 해야하므로
                {"$sort" : {"_id":1}},                                                      # avg 할 값에 $ 붙여줘야 함
                {"$limit" : 10},
                {
                    "$project" : {
                        "_id" : 0,
                        "year" : "$_id",     # $group에서 "_id"를 year별로 묶었고, _id를 year로 바꿔서 결과를 읽기 쉽게 만든 것
                        "avgRating" : 1
                    }
                }
            ]                                                                            
        }
    }
]

movie_report = list(movies.aggregate(movies_report_Pipeline))
movie_report

[{'latest5': [{'title': 'The Land Beyond the Sunset', 'year': 1912},
   {'title': 'Winsor McCay, the Famous Cartoonist of the N.Y. Herald and His Moving Comics',
    'year': 1911},
   {'title': 'A Corner in Wheat', 'year': 1909},
   {'title': 'The Great Train Robbery', 'year': 1903},
   {'title': 'Blacksmith Scene', 'year': 1893}],
  'highRatedCount': [{'count': 1596}],
  'genresTop10': [{'_id': 'Drama', 'count': 13789},
   {'_id': 'Comedy', 'count': 7024},
   {'_id': 'Romance', 'count': 3665},
   {'_id': 'Crime', 'count': 2678},
   {'_id': 'Thriller', 'count': 2658},
   {'_id': 'Action', 'count': 2539},
   {'_id': 'Documentary', 'count': 2129},
   {'_id': 'Adventure', 'count': 2045},
   {'_id': 'Horror', 'count': 1703},
   {'_id': 'Biography', 'count': 1404}],
  'yearlyAvgRecent10': [{'avgRating': 4.9, 'year': 1891},
   {'avgRating': 6.2, 'year': 1893},
   {'avgRating': 6.8, 'year': 1894},
   {'avgRating': 5.9, 'year': 1896},
   {'avgRating': 7.4, 'year': 1903},
   {'avgRating': 6.6, 